# Гибридная рекомендательная система: EDA + Matrix Factorization (SVD) + Ансамбли моделей ML

В данном исследовании реализован полный цикл построения рекомендательной системы:
1. **Разведочный анализ данных (EDA):** Проверка пропусков, анализ распределения рейтингов, проблемы «длинного хвоста» (Long Tail), популярности жанров и активности пользователей.
2. **Латентные признаки (TruncatedSVD):** Сжатие разреженной матрицы взаимодействий (`CSR`) для получения скрытых векторов пользователей и фильмов.
3. **Feature Engineering и Корреляционный анализ:** Построение признакового описания пары `(User, Movie)` и проверка мультиколлинеарности.
4. **Сравнение 7 моделей ML:** Соревнование линейных, деревянных и градиентных моделей (`Ridge`, `Decision Tree`, `Random Forest`, `Extra Trees`, `Bagging`, `AdaBoost`, `HistGradientBoosting`).
5. **Визуализация метрик и Экспорт в Go:** Выгрузка артефактов (`model_artifacts.json`) для быстрого инференса на бэкенде.


In [1]:
import os
import time
import urllib.request
import zipfile
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix

# Модели и метрики scikit-learn
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    BaggingRegressor,
    AdaBoostRegressor,
    HistGradientBoostingRegressor,
)

# Настройка стиля графиков
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

print("Все библиотеки и модули визуализации успешно загружены!")


ModuleNotFoundError: No module named 'numpy'

## 1. Загрузка базы данных (Data Ingestion)
Поддерживается как полная база (2.5 млн оценок, 65к фильмов), так и компактная версия MovieLens.


In [ ]:
DATA_DIR = "data"

if not os.path.exists(os.path.join(DATA_DIR, "ratings.csv")):
    print("Локальная база не найдена. Скачиваем датасет MovieLens...")
    os.makedirs(DATA_DIR, exist_ok=True)
    url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
    zip_path = "movielens.zip"
    urllib.request.urlretrieve(url, zip_path)
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    
    extracted_folder = "ml-latest-small"
    for filename in os.listdir(extracted_folder):
        os.replace(os.path.join(extracted_folder, filename), os.path.join(DATA_DIR, filename))
    os.rmdir(extracted_folder)
    os.remove(zip_path)
    print("База успешно скачана в папку 'data/'!")
else:
    print("Используется локальная база данных из папки 'data/'!")

ratings = pd.read_csv(
    os.path.join(DATA_DIR, "ratings.csv"),
    dtype={"userId": np.int32, "movieId": np.int32, "rating": np.float32}
)
movies = pd.read_csv(os.path.join(DATA_DIR, "movies.csv"))

print(f"Загружено оценок: {len(ratings):,}")
print(f"Загружено фильмов: {len(movies):,}")


## 2. Разведочный анализ данных (EDA — Exploratory Data Analysis)
Перед обучением моделей проведем комплексный анализ датасета:
1. **Целостность данных:** проверка на пропущенные значения (`NaN`), дубликаты и аномалии.
2. **Разреженность (Sparsity):** какую долю от всех возможных пар `(User, Movie)` составляют реальные оценки.
3. **Визуальный анализ:** распределение оценок, эффект «Длинного хвоста» (Long Tail), топ жанров и соотношение количества оценок со средним рейтингом фильма.


In [ ]:
print("=== 1. ПРОВЕРКА ПРОПУСКОВ И ДУБЛИКАТОВ ===")
print("Пропуски в ratings:\n", ratings.isnull().sum().to_dict())
print("Пропуски в movies:\n", movies.isnull().sum().to_dict())
print(f"Полных дубликатов в ratings: {ratings.duplicated(subset=['userId', 'movieId']).sum()}")

n_users = ratings["userId"].nunique()
n_movies = ratings["movieId"].nunique()
sparsity = (1.0 - len(ratings) / (n_users * n_movies)) * 100

print("\n=== 2. КЛЮЧЕВЫЕ ПАРАМЕТРЫ МАТРИЦЫ ВЗАИМОДЕЙСТВИЙ ===")
print(f"Уникальных пользователей: {n_users:,}")
print(f"Уникальных оцененных фильмов: {n_movies:,}")
print(f"Разреженность матрицы (Sparsity): {sparsity:.2f}% (именно поэтому необходимы CSR-матрицы и SVD)")

print("\n=== 3. ОПИСАТЕЛЬНАЯ СТАТИСТИКА (DESCRIBE) ===")
display(ratings[["rating"]].describe().T)


In [ ]:
# Подготовка агрегатов для визуализации EDA
movie_agg = ratings.groupby("movieId")["rating"].agg(["count", "mean"])
user_activity = ratings.groupby("userId")["rating"].count()

# Разбиваем жанры (разделены символом '|')
genres_exploded = movies["genres"].str.split("|").explode()
top_genres = genres_exploded[genres_exploded != "(no genres listed)"].value_counts().head(10)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# График 1: Распределение целевой переменной (Rating)
rating_counts = ratings["rating"].value_counts().sort_index()
sns.barplot(x=rating_counts.index.astype(str), y=rating_counts.values, ax=axes[0, 0], color="#4C72B0")
axes[0, 0].axvline(x=5, color="red", linestyle="--", alpha=0.6, label=f"Среднее: {ratings['rating'].mean():.2f}")
axes[0, 0].set_title("1. Распределение оценок пользователей", fontsize=13, fontweight="bold")
axes[0, 0].set_xlabel("Оценка (Rating)")
axes[0, 0].set_ylabel("Количество оценок")
axes[0, 0].legend()

# График 2: Эффект «Длинного хвоста» (Long Tail) популярности фильмов
sorted_counts = movie_agg["count"].sort_values(ascending=False).values
axes[0, 1].plot(sorted_counts, color="#DD8452", linewidth=2)
axes[0, 1].fill_between(range(len(sorted_counts)), sorted_counts, color="#DD8452", alpha=0.3)
axes[0, 1].set_yscale("log")
axes[0, 1].set_title("2. Эффект 'Длинного хвоста' (Лог. шкала оценок на фильм)", fontsize=13, fontweight="bold")
axes[0, 1].set_xlabel("Фильмы (отсортированы по популярности)")
axes[0, 1].set_ylabel("Число оценок (log scale)")

# График 3: Топ-10 самых популярных жанров в базе
sns.barplot(x=top_genres.values, y=top_genres.index, ax=axes[1, 0], palette="crest", hue=top_genres.index, legend=False)
axes[1, 0].set_title("3. Топ-10 жанров по количеству фильмов", fontsize=13, fontweight="bold")
axes[1, 0].set_xlabel("Количество фильмов")
axes[1, 0].set_ylabel("Жанр")

# График 4: Зависимость среднего рейтинга фильма от числа его оценок
sample_movies_agg = movie_agg.sample(min(3000, len(movie_agg)), random_state=42)
sns.scatterplot(data=sample_movies_agg, x="count", y="mean", alpha=0.4, ax=axes[1, 1], color="#55A868")
axes[1, 1].set_xscale("log")
axes[1, 1].set_title("4. Средний рейтинг фильма vs Популярность (log)", fontsize=13, fontweight="bold")
axes[1, 1].set_xlabel("Количество оценок у фильма (log scale)")
axes[1, 1].set_ylabel("Средний рейтинг фильма")

plt.tight_layout()
plt.savefig("eda_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Вывод по EDA: Наблюдается сильная асимметрия популярности (Long Tail) и смещение оценок к 3.5–4.0.")


## 3. Построение разреженной матрицы и обучение TruncatedSVD
Извлекаем латентные (скрытые) факторы пользователей и фильмов из разреженной матрицы `CSR`.


In [ ]:
unique_users = ratings["userId"].unique()
unique_movies = ratings["movieId"].unique()

user_to_idx = {int(uid): idx for idx, uid in enumerate(unique_users)}
movie_to_idx = {int(mid): idx for idx, mid in enumerate(unique_movies)}
idx_to_user = {idx: int(uid) for uid, idx in user_to_idx.items()}
idx_to_movie = {idx: int(mid) for mid, idx in movie_to_idx.items()}

ratings["u_idx"] = ratings["userId"].map(user_to_idx)
ratings["m_idx"] = ratings["movieId"].map(movie_to_idx)

sparse_user_item = csr_matrix(
    (ratings["rating"].values, (ratings["u_idx"].values, ratings["m_idx"].values)),
    shape=(len(unique_users), len(unique_movies)),
    dtype=np.float32
)

N_COMPONENTS = 16
svd_model = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
user_factors = svd_model.fit_transform(sparse_user_item)
movie_factors = svd_model.components_.T

print(f"SVD обучен! Профили пользователей: {user_factors.shape}, Профили фильмов: {movie_factors.shape}")


## 4. Feature Engineering и Матрица корреляций признаков
Формируем матрицу признаков $X$ и проверяем связь сконструированных признаков с целевой переменной `rating` через тепловую карту корреляций Пирсона (`Heatmap`).


In [ ]:
MAX_SAMPLES = 200_000
if len(ratings) > MAX_SAMPLES:
    df_sample = ratings.sample(n=MAX_SAMPLES, random_state=42).copy()
    print(f"Взята подвыборка из {MAX_SAMPLES:,} оценок для быстрого обучения ансамблей.")
else:
    df_sample = ratings.copy()

user_stats = ratings.groupby("u_idx")["rating"].agg(["mean", "count"]).rename(columns={"mean": "u_mean", "count": "u_count"})
movie_stats = ratings.groupby("m_idx")["rating"].agg(["mean", "count"]).rename(columns={"mean": "m_mean", "count": "m_count"})

df_sample = df_sample.merge(user_stats, on="u_idx", how="left")
df_sample = df_sample.merge(movie_stats, on="m_idx", how="left")

u_vecs = user_factors[df_sample["u_idx"].values]
m_vecs = movie_factors[df_sample["m_idx"].values]
svd_dot = np.sum(u_vecs * m_vecs, axis=1, keepdims=True)
df_sample["svd_dot"] = svd_dot.flatten()

# Строим матрицу корреляций ключевых признаков с целевым рейтингом
plt.figure(figsize=(8, 5))
corr_cols = ["rating", "u_mean", "m_mean", "svd_dot", "u_count", "m_count"]
sns.heatmap(df_sample[corr_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Матрица корреляций ключевых признаков с целевой оценкой (rating)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Собираем финальную матрицу X и вектор y
stat_features = df_sample[["u_mean", "u_count", "m_mean", "m_count"]].values
X = np.hstack([stat_features, svd_dot, u_vecs[:, :8], m_vecs[:, :8]]).astype(np.float32)
y = df_sample["rating"].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Размер матрицы признаков X_train: {X_train.shape}, X_test: {X_test.shape}")


## 5. Обучение и сравнение 7 моделей машинного обучения
Сравниваем 7 алгоритмов регрессии по метрикам **RMSE**, **MAE**, **R²** и времени обучения.


In [ ]:
PARAMS = {
    "Ridge":             dict(alpha=1.0, random_state=42),
    "Decision Tree":     dict(max_depth=12, min_samples_leaf=25, random_state=42),
    "Random Forest":     dict(n_estimators=80, max_depth=16, min_samples_leaf=10, max_features=0.6,
                              bootstrap=True, max_samples=0.5, n_jobs=-1, random_state=42),
    "Extra Trees":       dict(n_estimators=80, max_depth=18, min_samples_leaf=10, max_features=0.7,
                              bootstrap=True, max_samples=0.5, n_jobs=-1, random_state=42),
    "Bagging":           dict(n_estimators=30, max_samples=0.5, n_jobs=-1, random_state=42),
    "AdaBoost":          dict(n_estimators=50, learning_rate=0.3, random_state=42),
    "Gradient Boosting": dict(max_iter=400, learning_rate=0.06, max_leaf_nodes=63,
                              min_samples_leaf=30, l2_regularization=1.0, early_stopping=True, random_state=42),
}

MODELS = {
    "Ridge":             Ridge(**PARAMS["Ridge"]),
    "Decision Tree":     DecisionTreeRegressor(**PARAMS["Decision Tree"]),
    "Random Forest":     RandomForestRegressor(**PARAMS["Random Forest"]),
    "Extra Trees":       ExtraTreesRegressor(**PARAMS["Extra Trees"]),
    "Bagging":           BaggingRegressor(**PARAMS["Bagging"]),
    "AdaBoost":          AdaBoostRegressor(**PARAMS["AdaBoost"]),
    "Gradient Boosting": HistGradientBoostingRegressor(**PARAMS["Gradient Boosting"]),
}

results = []
trained_models = {}

for name, model in MODELS.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0
    
    preds = np.clip(model.predict(X_test), 0.5, 5.0)
    
    rmse = root_mean_squared_error(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    trained_models[name] = model
    results.append({
        "Model": name,
        "RMSE": round(rmse, 4),
        "MAE": round(mae, 4),
        "R2": round(r2, 4),
        "Time (sec)": round(train_time, 2)
    })
    print(f"{name:<18} | RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f} | Time: {train_time:.2f}s")

leaderboard = pd.DataFrame(results).sort_values(by="RMSE", ascending=True).reset_index(drop=True)
print("\n=== ИТОГОВЫЙ РЕЙТИНГ МОДЕЛЕЙ (Сортировка по RMSE) ===")
display(leaderboard)


## 6. Визуализация сравнения моделей
Строим итоговые графики сравнения всех 7 обученных алгоритмов по ошибке **RMSE**, метрике **MAE** и соотношению **Качество ($R^2$) vs Время обучения**.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# График 1: Сравнение RMSE моделей
sns.barplot(data=leaderboard, x="RMSE", y="Model", ax=axes[0], palette="viridis", hue="Model", legend=False)
axes[0].set_title("1. Ошибка моделей RMSE (меньше = лучше)", fontsize=13, fontweight="bold")
axes[0].set_xlim(leaderboard["RMSE"].min() * 0.9, leaderboard["RMSE"].max() * 1.05)
for i, v in enumerate(leaderboard["RMSE"]):
    axes[0].text(v + 0.003, i, f"{v:.4f}", va="center", fontsize=10, fontweight="bold")

# График 2: Точность (R2) vs Время обучения (Time)
sns.scatterplot(data=leaderboard, x="Time (sec)", y="R2", hue="Model", s=180, ax=axes[1], palette="deep")
for _, row in leaderboard.iterrows():
    axes[1].text(row["Time (sec)"] + 0.02, row["R2"], row["Model"], fontsize=9, va="center")
axes[1].set_title("2. Качество (R²) vs Время обучения (сек)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Время обучения (секунды)")
axes[1].set_ylabel("Коэффициент детерминации R² (больше = лучше)")
axes[1].legend().remove()

plt.tight_layout()
plt.savefig("models_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("График сохранен в файл 'models_comparison.png'!")


## 7. Экспорт лучшей модели и артефактов для Go-проекта
Сохраняем векторы и таблицу метрик в `model_artifacts.json`.


In [ ]:
best_model_name = leaderboard.iloc[0]["Model"]
print(f"Лучшая модель по итогам экспериментов: {best_model_name} (RMSE = {leaderboard.iloc[0]['RMSE']})")

export_movies = {
    str(idx_to_movie[idx]): np.round(vec, 4).tolist()
    for idx, vec in enumerate(movie_factors)
}
export_users = {
    str(idx_to_user[idx]): np.round(vec, 4).tolist()
    for idx, vec in enumerate(user_factors[:1000])
}

artifacts = {
    "best_model": best_model_name,
    "metrics_leaderboard": results,
    "n_components": N_COMPONENTS,
    "movies": export_movies,
    "users": export_users
}

with open("model_artifacts.json", "w", encoding="utf-8") as f:
    json.dump(artifacts, f)

print("Файл 'model_artifacts.json' успешно сохранен!")
